In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

## Step 1. Dataset 생성

In [2]:
# ---------------------------
# 1. Single state generator
# ---------------------------

def generate_equator_state(eta):
    """
    Generate |psi(eta)> = (|0> + e^{i eta} |1>) / sqrt(2)
    Returns statevector of shape (2,)
    """
    return np.array([
        1/np.sqrt(2),
        np.exp(1j * eta) / np.sqrt(2)
    ], dtype=complex)


# ---------------------------
# 2. Batch generator
# ---------------------------

def generate_batch(batch_size):
    """
    Generate a batch of equator states.
    Returns:
        etas: (batch_size,)
        states: (batch_size, 2)
    """
    etas = np.random.uniform(0, 2*np.pi, batch_size)
    states = np.array([generate_equator_state(eta) for eta in etas])
    return etas, states


# ---------------------------
# 3. Full dataset generator
# ---------------------------

def generate_dataset(num_samples=30, train_ratio=0.8):
    """
    Reproduce paper setting:
    - 30 samples
    - 80-20 train-test split
    """
    etas = np.random.uniform(0, 2*np.pi, num_samples)
    states = np.array([generate_equator_state(eta) for eta in etas])

    split_idx = int(train_ratio * num_samples)

    train_states = states[:split_idx]
    test_states = states[split_idx:]

    train_etas = etas[:split_idx]
    test_etas = etas[split_idx:]

    return (train_etas, train_states), (test_etas, test_states)

In [3]:
etas, states = generate_batch(5)

print("etas:", etas)
print("state shape:", states.shape)
print("first state:", states[0])
print("norm check:", np.linalg.norm(states[0]))

etas: [5.48244636 4.81493298 3.85435342 2.5206916  1.15843616]
state shape: (5, 2)
first state: [0.70710678+0.j         0.49227108-0.50761126j]
norm check: 0.9999999999999999


In [7]:
(train_etas, train_states), (test_etas, test_states) = generate_dataset()
train_etas.shape, train_states.shape, test_etas.shape, test_states.shape

((24,), (24, 2), (6,), (6, 2))

## Step 2. 양자 회로

In [12]:
import pennylane as qml
import numpy as np

dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def cloning_circuit(params, eta):

    # wires: 0=A(input), 1=E, 2=E*

    # 1️⃣ Input state preparation
    qml.Hadamard(wires=0)
    qml.RZ(eta, wires=0)

    # 2️⃣ Variational block (Fig 2b)

    # R_y(alpha1) on qubit 1
    qml.RY(params[0], wires=1)

    # CNOT 1→2
    qml.CNOT(wires=[1,2])

    # R_y(alpha2) on qubit 2
    qml.RY(params[1], wires=2)

    # CNOT 2→1
    qml.CNOT(wires=[2,1])

    # R_y(alpha3) on qubit 1
    qml.RY(params[2], wires=1)

    # CNOTs
    qml.CNOT(wires=[0,1])
    qml.CNOT(wires=[0,2])
    qml.CNOT(wires=[1,0])
    qml.CNOT(wires=[2,0])

    return qml.density_matrix(wires=[0,1,2])

In [15]:
def get_reduced_density_matrices(rho_full):
    rho_B = qml.math.reduce_dm(rho_full, indices=[1])
    rho_E = qml.math.reduce_dm(rho_full, indices=[2])
    return rho_B, rho_E

In [16]:
import numpy as np

# ---------------------------
# Test parameters
# ---------------------------

np.random.seed(42)

params = np.random.uniform(0, 2*np.pi, 3)
eta = np.random.uniform(0, 2*np.pi)

# ---------------------------
# Run circuit
# ---------------------------

rho_full = cloning_circuit(params, eta)

print("Full density matrix shape:", rho_full.shape)

# ---------------------------
# Reduced density matrices
# ---------------------------

rho_B, rho_E = get_reduced_density_matrices(rho_full)

print("rho_B shape:", rho_B.shape)
print("rho_E shape:", rho_E.shape)

# ---------------------------
# Sanity checks
# ---------------------------

print("\nTrace checks:")
print("Tr(rho_full) =", np.trace(rho_full))
print("Tr(rho_B) =", np.trace(rho_B))
print("Tr(rho_E) =", np.trace(rho_E))

print("\nHermitian check (rho_B - rho_B†):")
print(np.linalg.norm(rho_B - rho_B.conj().T))

print("\nHermitian check (rho_E - rho_E†):")
print(np.linalg.norm(rho_E - rho_E.conj().T))

Full density matrix shape: (8, 8)
rho_B shape: (2, 2)
rho_E shape: (2, 2)

Trace checks:
Tr(rho_full) = (0.9999999999999996+0j)
Tr(rho_B) = (0.9999999999999996+0j)
Tr(rho_E) = (0.9999999999999996+0j)

Hermitian check (rho_B - rho_B†):
1.3342040676371933e-17

Hermitian check (rho_E - rho_E†):
2.072184654912181e-17


In [18]:
import pennylane as qml
import numpy as np

# =====================================
# 1️⃣ Device
# =====================================

dev = qml.device("default.qubit", wires=3)

# =====================================
# 2️⃣ Cloning Circuit (Fig.2b)
# =====================================

@qml.qnode(dev, interface="autograd")
def cloning_circuit(params, eta):
    """
    wires:
    0 = A (input)
    1 = Bob
    2 = Eve
    """

    # Input state |psi(eta)> = H -> RZ(eta) |0>
    qml.Hadamard(wires=0)
    qml.RZ(eta, wires=0)

    # Variational block
    qml.RY(params[0], wires=1)
    qml.CNOT(wires=[1,2])
    qml.RY(params[1], wires=2)
    qml.CNOT(wires=[2,1])
    qml.RY(params[2], wires=1)

    # Cloning interaction block
    qml.CNOT(wires=[0,1])
    qml.CNOT(wires=[0,2])
    qml.CNOT(wires=[1,0])
    qml.CNOT(wires=[2,0])

    return qml.density_matrix(wires=[0,1,2])


# =====================================
# 3️⃣ Fidelity
# =====================================

def fidelity_from_density(rho, eta):
    """
    F = ⟨psi| rho |psi⟩
    """
    psi = qml.math.stack([
        1/qml.math.sqrt(2),
        qml.math.exp(1j * eta)/qml.math.sqrt(2)
    ])

    psi_dag = qml.math.conj(psi)
    return qml.math.real(psi_dag @ rho @ psi)


# =====================================
# 4️⃣ Cost (Squared Local)
# =====================================

def cloning_cost(params, eta):

    rho_full = cloning_circuit(params, eta)

    rho_B = qml.math.reduce_dm(rho_full, indices=[1])
    rho_E = qml.math.reduce_dm(rho_full, indices=[2])

    F_B = fidelity_from_density(rho_B, eta)
    F_E = fidelity_from_density(rho_E, eta)

    return (
        (1 - F_B)**2
        + (1 - F_E)**2
        + (F_B - F_E)**2
    )


def batch_cost(params, etas):
    costs = [cloning_cost(params, eta) for eta in etas]
    return qml.math.mean(qml.math.stack(costs))


# =====================================
# 5️⃣ Dataset Generation
# =====================================

def generate_dataset(num_samples=30, train_ratio=0.8):
    etas = np.random.uniform(0, 2*np.pi, num_samples)
    split = int(train_ratio * num_samples)
    return etas[:split], etas[split:]


# =====================================
# 6️⃣ Training Loop
# =====================================

np.random.seed(42)

train_etas, test_etas = generate_dataset()

# Initialize parameters
params = qml.numpy.array(
    np.random.uniform(0, 2*np.pi, 3),
    requires_grad=True
)

optimizer = qml.AdamOptimizer(stepsize=0.05)

epochs = 2000
batch_size = 10

print("Initial params:", params)

for epoch in range(epochs):

    # mini-batch sampling
    batch_indices = np.random.choice(len(train_etas), batch_size, replace=False)
    batch = train_etas[batch_indices]

    params = optimizer.step(lambda p: batch_cost(p, batch), params)

    if epoch % 20 == 0:
        train_loss = batch_cost(params, train_etas)
        print(f"Epoch {epoch:3d} | Train Cost: {train_loss:.6f}")

print("\nTraining complete.")
print("Final params:", params)


# =====================================
# 7️⃣ Final Fidelity Check
# =====================================

def evaluate_fidelity(params, etas):

    F_B_list = []
    F_E_list = []

    for eta in etas:
        rho_full = cloning_circuit(params, eta)
        rho_B = qml.math.reduce_dm(rho_full, indices=[1])
        rho_E = qml.math.reduce_dm(rho_full, indices=[2])

        F_B_list.append(fidelity_from_density(rho_B, eta))
        F_E_list.append(fidelity_from_density(rho_E, eta))

    return np.mean(F_B_list), np.mean(F_E_list)


F_B_avg, F_E_avg = evaluate_fidelity(params, test_etas)

print("\nAverage Test Fidelity:")
print("Bob :", F_B_avg)
print("Eve :", F_E_avg)

Initial params: [3.81731689 1.07143467 0.40873121]
Epoch   0 | Train Cost: 0.592217
Epoch  20 | Train Cost: 0.506740
Epoch  40 | Train Cost: 0.500832
Epoch  60 | Train Cost: 0.500111
Epoch  80 | Train Cost: 0.500002
Epoch 100 | Train Cost: 0.500003
Epoch 120 | Train Cost: 0.500000
Epoch 140 | Train Cost: 0.500000
Epoch 160 | Train Cost: 0.500000
Epoch 180 | Train Cost: 0.500000
Epoch 200 | Train Cost: 0.500000
Epoch 220 | Train Cost: 0.500000
Epoch 240 | Train Cost: 0.500000
Epoch 260 | Train Cost: 0.500000
Epoch 280 | Train Cost: 0.500000
Epoch 300 | Train Cost: 0.500000
Epoch 320 | Train Cost: 0.500000
Epoch 340 | Train Cost: 0.500000
Epoch 360 | Train Cost: 0.500000
Epoch 380 | Train Cost: 0.500000
Epoch 400 | Train Cost: 0.500000
Epoch 420 | Train Cost: 0.500000
Epoch 440 | Train Cost: 0.500000
Epoch 460 | Train Cost: 0.500000
Epoch 480 | Train Cost: 0.500000
Epoch 500 | Train Cost: 0.500000
Epoch 520 | Train Cost: 0.500000
Epoch 540 | Train Cost: 0.500000
Epoch 560 | Train Cost: 0

In [19]:
params

tensor([3.57781512, 1.5789493 , 0.43572766], requires_grad=True)